## ETL Silver → Gold
É o processo que pega os dados já tratados e padronizados da camada Silver e os transforma na camada Gold, que é a camada “final” para consumo. O objetivo é uma modelagem organizada com dados em estruturas prontas para análise e garantir que o resultado fique consistente, performático e fácil de consultar por dashboards e relatórios.

In [1]:
from __future__ import annotations

import os
import warnings

import numpy as np
import pandas as pd
from sqlalchemy import create_engine, text

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

## Célula 2 — Conexão e DDL da DW
Preparamos a camada Gold (dw) estabelecendo conexão com o PostgreSQL via SQLAlchemy e aplicando o ddl.sql no schema dw para recriar as estruturas do Data Warehouse (dimensões e fato). O script é localizado em caminhos previstos do projeto, validado com erro explícito caso não exista, e executado em transação, garantindo que a base fique pronta antes das cargas de dados do ETL.

In [2]:
POSTGRES_DB = os.getenv("POSTGRES_DB", "airline_delay_causes")
POSTGRES_USER = os.getenv("POSTGRES_USER", "postgres")
POSTGRES_PASSWORD = os.getenv("POSTGRES_PASSWORD", "postgres")
POSTGRES_HOST = os.getenv("POSTGRES_HOST", "localhost")
POSTGRES_PORT = int(os.getenv("POSTGRES_PORT", "5432"))

DW_SCHEMA = "dw"

connection_string = (
    f"postgresql+psycopg2://{POSTGRES_USER}:{POSTGRES_PASSWORD}"
    f"@{POSTGRES_HOST}:{POSTGRES_PORT}/{POSTGRES_DB}"
)
engine = create_engine(connection_string)

print("=" * 80)
print("ETAPA 1: DDL - Criar estruturas da DW")
print("=" * 80)

ddl_candidates = [
    "../Data Layer/gold/ddl.sql",
    "../Data Layer/dw/ddl.sql",
    "./ddl.sql",
]

ddl_path = next((p for p in ddl_candidates if os.path.exists(p)), None)
if ddl_path is None:
    raise FileNotFoundError(
        "Não encontrei o arquivo ddl.sql. Tentei estes caminhos:\n"
        + "\n".join(f"- {p}" for p in ddl_candidates)
        + "\n\nDica: rode este notebook a partir da pasta Transformer (como nos outros ETLs)."
    )

with open(ddl_path, "r", encoding="utf-8") as f:
    ddl_sql = f.read()

# Executar o script (multi-statements)
# Estratégia: quebrar por ';' e executar um a um (robusto no Jupyter)
statements = [s.strip() for s in ddl_sql.split(";") if s.strip()]

with engine.begin() as conn:
    for stmt in statements:
        conn.execute(text(stmt))

print(f"DDL aplicado com sucesso: {ddl_path}")


ETAPA 1: DDL - Criar estruturas da DW
DDL aplicado com sucesso: ../Data Layer/gold/ddl.sql


# Célula 3 — EXTRACT da Silver
Realizamos a etapa de extração do ETL Silver → Gold, carregando integralmente a tabela silver.silver_airline_on_time para um DataFrame (df_silver) que servirá de base para a modelagem e carga da DW. Imprime dimensões e amostra inicial para inspeção rápida, e executa checagens objetivas de qualidade: volume de nulos por coluna e contagem de duplicidades no grão esperado da Silver (year, month, carrier, airport).

In [3]:
print("=" * 80)
print("ETAPA 2: EXTRACT - Ler dados da SILVER")
print("=" * 80)

sql_silver = "SELECT * FROM silver.silver_airline_on_time;"
df_silver = pd.read_sql_query(sql_silver, engine)

print("Linhas, colunas (Silver):", df_silver.shape)
display(df_silver.head(5))

# Checagens rápidas (visibilidade)
print("\nNulos por coluna (top 10):")
display(df_silver.isna().sum().sort_values(ascending=False).head(10))

print("\nDuplicados no grão da Silver (year, month, carrier, airport):")
dup = int(df_silver.duplicated(subset=["year", "month", "carrier", "airport"]).sum())
print(dup)


ETAPA 2: EXTRACT - Ler dados da SILVER
Linhas, colunas (Silver): (279182, 23)


,year,month,carrier,carrier_name,airport,airport_name,arr_flights,arr_del15,carrier_ct,weather_ct,nas_ct,security_ct,late_aircraft_ct,arr_cancelled,arr_diverted,arr_delay,carrier_delay,weather_delay,nas_delay,security_delay,late_aircraft_delay,is_outlier_arr_delay,flight_date
0,2004,1,DL,Delta Air Lines Inc.,PBI,"West Palm Beach/Palm Beach, FL: Palm Beach Int...",650,126,21,6,52,1,46,4,0,5425.0,881.0,397.0,2016.0,15.0,2116.0,0,2004-01-01
1,2004,1,DL,Delta Air Lines Inc.,PDX,"Portland, OR: Portland International",314,61,14,3,34,0,10,30,3,2801.0,478.0,239.0,1365.0,0.0,719.0,0,2004-01-01
2,2004,1,DL,Delta Air Lines Inc.,PHL,"Philadelphia, PA: Philadelphia International",513,97,28,0,52,0,17,15,0,4261.0,1150.0,16.0,2286.0,0.0,809.0,0,2004-01-01
3,2004,1,DL,Delta Air Lines Inc.,PHX,"Phoenix, AZ: Phoenix Sky Harbor International",334,78,20,2,39,0,16,3,1,3400.0,1159.0,166.0,1295.0,0.0,780.0,0,2004-01-01
4,2004,1,DL,Delta Air Lines Inc.,PIT,"Pittsburgh, PA: Pittsburgh International",217,47,8,0,22,0,17,4,1,1737.0,350.0,28.0,522.0,0.0,837.0,0,2004-01-01



Nulos por coluna (top 10):


year            0
month           0
carrier         0
carrier_name    0
airport         0
airport_name    0
arr_flights     0
arr_del15       0
carrier_ct      0
weather_ct      0
dtype: int64


Duplicados no grão da Silver (year, month, carrier, airport):
0


## Célula 4 — TRANSFORM das dimensões
Constrói as dimensões da DW a partir da Silver, padronizando códigos de companhia e aeroporto para garantir consistência de chaves e joins. Em seguida, gera dim_tmp, dim_cia e dim_apt no grão exigido pelo DDL, incluindo uma data âncora no primeiro dia do mês para evitar violação de unicidade temporal e resolvendo conflitos de nomes por critério de ocorrência mais frequente. Ao final, executa validações mínimas (nulos e duplicidades) para prevenir falhas na carga por restrições de PK/UNIQUE.

In [4]:
# Observação:
# A Silver é mensal, mas pode ter várias datas dentro do mesmo mês.
# Para não quebrar o UNIQUE (num_ano,num_mes), usamos dat_tmp como "âncora": 1º dia do mês.

print("=" * 80)
print("ETAPA 3: TRANSFORM - Montar DIMs (DataFrames)")
print("=" * 80)

df = df_silver.copy()

# Padronizações para consistência (principalmente joins)
df["carrier"] = df["carrier"].astype(str).str.strip().str.upper()
df["airport"] = df["airport"].astype(str).str.strip().str.upper()

# DIM_TMP: 1 linha por (ano,mês)
dim_tmp = (
    df[["year", "month"]]
    .drop_duplicates()
    .rename(columns={"year": "num_ano", "month": "num_mes"})
)

# dat_tmp = 1º dia do mês (âncora)
dim_tmp["dat_tmp"] = pd.to_datetime(
    dim_tmp["num_ano"].astype(str) + "-" + dim_tmp["num_mes"].astype(str) + "-01",
    errors="coerce"
).dt.date

# trimestre (1..12 -> 1..4)
dim_tmp["num_tri"] = ((dim_tmp["num_mes"] - 1) // 3 + 1).astype("Int64")

# nome do mês (opcional)
mes_pt = {
    1: "Janeiro", 2: "Fevereiro", 3: "Março", 4: "Abril",
    5: "Maio", 6: "Junho", 7: "Julho", 8: "Agosto",
    9: "Setembro", 10: "Outubro", 11: "Novembro", 12: "Dezembro"
}
dim_tmp["nom_mes"] = dim_tmp["num_mes"].map(mes_pt)

dim_tmp = dim_tmp.sort_values(["num_ano", "num_mes"]).reset_index(drop=True)

print("dim_tmp (linhas, colunas):", dim_tmp.shape)
display(dim_tmp.head(5))
print("Duplicados (dim_tmp) em (num_ano,num_mes):", int(dim_tmp.duplicated(subset=["num_ano", "num_mes"]).sum()))

# DIM_CIA: 1 linha por cod_cia; nome MAIS FREQUENTE em conflitos
tmp_cia = (
    df[["carrier", "carrier_name"]]
    .rename(columns={"carrier": "cod_cia", "carrier_name": "nom_cia"})
    .copy()
)
tmp_cia["nom_cia"] = tmp_cia["nom_cia"].astype(str).str.strip()
tmp_cia.loc[tmp_cia["nom_cia"].isin(["", "None", "nan", "NaN"]), "nom_cia"] = np.nan

cia_freq = (
    tmp_cia
    .dropna(subset=["cod_cia"])
    .groupby(["cod_cia", "nom_cia"], dropna=False)
    .size()
    .reset_index(name="qtd_occ")
)

dim_cia = (
    cia_freq
    .sort_values(["cod_cia", "qtd_occ", "nom_cia"], ascending=[True, False, True])
    .drop_duplicates(subset=["cod_cia"], keep="first")
    .drop(columns=["qtd_occ"])
    .reset_index(drop=True)
)

print("\ndim_cia (linhas, colunas):", dim_cia.shape)
display(dim_cia.head(5))
print("Duplicados (dim_cia) em cod_cia:", int(dim_cia.duplicated(subset=["cod_cia"]).sum()))

# DIM_APT: 1 linha por cod_apt; nome MAIS FREQUENTE em conflitos
tmp_apt = (
    df[["airport", "airport_name"]]
    .rename(columns={"airport": "cod_apt", "airport_name": "nom_apt"})
    .copy()
)
tmp_apt["nom_apt"] = tmp_apt["nom_apt"].astype(str).str.strip()
tmp_apt.loc[tmp_apt["nom_apt"].isin(["", "None", "nan", "NaN"]), "nom_apt"] = np.nan

apt_freq = (
    tmp_apt
    .dropna(subset=["cod_apt"])
    .groupby(["cod_apt", "nom_apt"], dropna=False)
    .size()
    .reset_index(name="qtd_occ")
)

dim_apt = (
    apt_freq
    .sort_values(["cod_apt", "qtd_occ", "nom_apt"], ascending=[True, False, True])
    .drop_duplicates(subset=["cod_apt"], keep="first")
    .drop(columns=["qtd_occ"])
    .reset_index(drop=True)
)

print("\ndim_apt (linhas, colunas):", dim_apt.shape)
display(dim_apt.head(5))
print("Duplicados (dim_apt) em cod_apt:", int(dim_apt.duplicated(subset=["cod_apt"]).sum()))

# Checagens mínimas (evita falha no INSERT)
if dim_tmp[["num_ano", "num_mes", "dat_tmp"]].isna().any().any():
    raise ValueError("DIM_TMP tem nulos em (num_ano, num_mes, dat_tmp).")
if dim_cia[["cod_cia"]].isna().any().any():
    raise ValueError("DIM_CIA tem nulos em cod_cia.")
if dim_apt[["cod_apt"]].isna().any().any():
    raise ValueError("DIM_APT tem nulos em cod_apt.")


ETAPA 3: TRANSFORM - Montar DIMs (DataFrames)
dim_tmp (linhas, colunas): (202, 5)


,num_ano,num_mes,dat_tmp,num_tri,nom_mes
0,2003,6,2003-06-01,2,Junho
1,2003,7,2003-07-01,3,Julho
2,2003,8,2003-08-01,3,Agosto
3,2003,9,2003-09-01,3,Setembro
4,2003,10,2003-10-01,4,Outubro


Duplicados (dim_tmp) em (num_ano,num_mes): 0

dim_cia (linhas, colunas): (28, 2)


,cod_cia,nom_cia
0,9E,Pinnacle Airlines Inc.
1,AA,American Airlines Inc.
2,AQ,Aloha Airlines Inc.
3,AS,Alaska Airlines Inc.
4,B6,JetBlue Airways


Duplicados (dim_cia) em cod_cia: 0

dim_apt (linhas, colunas): (409, 2)


,cod_apt,nom_apt
0,ABE,"Allentown/Bethlehem/Easton, PA: Lehigh Valley ..."
1,ABI,"Abilene, TX: Abilene Regional"
2,ABQ,"Albuquerque, NM: Albuquerque International Sun..."
3,ABR,"Aberdeen, SD: Aberdeen Regional"
4,ABY,"Albany, GA: Southwest Georgia Regional"


Duplicados (dim_apt) em cod_apt: 0


## Célula 5 — LOAD das dimensões na DW

Executa a carga das dimensões no schema dw, garantindo reprocessamento seguro. Primeiro, faz TRUNCATE conjunto de fato e dimensões com reinício de identidade para respeitar dependências de FK e evitar violações de UNIQUE em execuções repetidas. Em seguida, insere dim_tmp, dim_cia e dim_apt em lote via to_sql para desempenho. Por fim, valida a carga com contagem de registros por dimensão.

In [5]:
print("=" * 80)
print("ETAPA 4: LOAD - Inserir DIMs na DW")
print("=" * 80)

# Limpar tabelas (truncar em conjunto por causa das FKs)
with engine.begin() as conn:
    conn.execute(text(
        f"TRUNCATE TABLE {DW_SCHEMA}.fat_atr, {DW_SCHEMA}.dim_tmp, {DW_SCHEMA}.dim_cia, {DW_SCHEMA}.dim_apt "
        "RESTART IDENTITY;"
    ))

# Inserções em batch via to_sql
dim_tmp.to_sql("dim_tmp", con=engine, schema=DW_SCHEMA, if_exists="append", index=False, method="multi", chunksize=5000)
dim_cia.to_sql("dim_cia", con=engine, schema=DW_SCHEMA, if_exists="append", index=False, method="multi", chunksize=5000)
dim_apt.to_sql("dim_apt", con=engine, schema=DW_SCHEMA, if_exists="append", index=False, method="multi", chunksize=5000)

# Conferência rápida
with engine.connect() as conn:
    qtd_tmp = conn.execute(text(f"SELECT COUNT(*) FROM {DW_SCHEMA}.dim_tmp;")).scalar()
    qtd_cia = conn.execute(text(f"SELECT COUNT(*) FROM {DW_SCHEMA}.dim_cia;")).scalar()
    qtd_apt = conn.execute(text(f"SELECT COUNT(*) FROM {DW_SCHEMA}.dim_apt;")).scalar()

print(f"Linhas na DW: dim_tmp={qtd_tmp}, dim_cia={qtd_cia}, dim_apt={qtd_apt}")


ETAPA 4: LOAD - Inserir DIMs na DW
Linhas na DW: dim_tmp=202, dim_cia=28, dim_apt=409


## Célula 6 — Montagem da fato com resolução de SRKs
Prepara a carga da tabela fato da DW convertendo as chaves naturais da Silver em SRKs. Para isso, lê as dimensões já carregadas no banco (com srk_*), padroniza as chaves para garantir correspondência e realiza merges para obter srk_tmp, srk_cia e srk_apt. Com as chaves resolvidas, monta o DataFrame final da dw.fat_atr no formato exato do DDL, aplicando conversões de tipo, tratamento de inf/NaN, proteção para não-negatividade conforme CHECKs e deduplicação no grão da fato (srk_tmp, srk_cia, srk_apt).

In [6]:
print("=" * 80)
print("ETAPA 5: TRANSFORM - Resolver SRKs e montar FATO (DataFrame)")
print("=" * 80)

# Ler dimensões do banco (com srk_*)
dim_tmp_db = pd.read_sql_query(f"SELECT srk_tmp, num_ano, num_mes FROM {DW_SCHEMA}.dim_tmp;", engine)
dim_cia_db = pd.read_sql_query(f"SELECT srk_cia, cod_cia FROM {DW_SCHEMA}.dim_cia;", engine)
dim_apt_db = pd.read_sql_query(f"SELECT srk_apt, cod_apt FROM {DW_SCHEMA}.dim_apt;", engine)

# Se estiver vazio, significa que você não rodou a CÉLULA 5 (ou falhou nela)
if dim_tmp_db.empty or dim_cia_db.empty or dim_apt_db.empty:
    raise ValueError(
        "DIMs vazias no schema DW. Rode a CÉLULA 5 (ETAPA 4) para carregar as DIMs antes da ETAPA 5.\n"
        f"- dim_tmp vazia? {dim_tmp_db.empty}\n"
        f"- dim_cia vazia? {dim_cia_db.empty}\n"
        f"- dim_apt vazia? {dim_apt_db.empty}"
    )

# Padronizar chaves NATURAIS do lado das DIMs (por segurança)
dim_cia_db["cod_cia"] = dim_cia_db["cod_cia"].astype(str).str.strip().str.upper()
dim_apt_db["cod_apt"] = dim_apt_db["cod_apt"].astype(str).str.strip().str.upper()
dim_tmp_db["num_ano"] = pd.to_numeric(dim_tmp_db["num_ano"], errors="coerce").astype("Int64")
dim_tmp_db["num_mes"] = pd.to_numeric(dim_tmp_db["num_mes"], errors="coerce").astype("Int64")

# Base para a fato (Silver)
df_fact_base = df_silver.copy()

# Padronizar chaves NATURAIS do lado da Silver
df_fact_base["year"] = pd.to_numeric(df_fact_base["year"], errors="coerce").astype("Int64")
df_fact_base["month"] = pd.to_numeric(df_fact_base["month"], errors="coerce").astype("Int64")
df_fact_base["carrier"] = df_fact_base["carrier"].astype(str).str.strip().str.upper()
df_fact_base["airport"] = df_fact_base["airport"].astype(str).str.strip().str.upper()

# Limpar inf/-inf (evita violar checks anti NaN/Inf do DDL)
df_fact_base = df_fact_base.replace([np.inf, -np.inf], np.nan)

# Resolver SRKs
df_fact_base = df_fact_base.merge(
    dim_tmp_db,
    left_on=["year", "month"],
    right_on=["num_ano", "num_mes"],
    how="left"
)

df_fact_base = df_fact_base.merge(
    dim_cia_db,
    left_on=["carrier"],
    right_on=["cod_cia"],
    how="left"
)

df_fact_base = df_fact_base.merge(
    dim_apt_db,
    left_on=["airport"],
    right_on=["cod_apt"],
    how="left"
)

# Validar SRKs
missing_tmp = int(df_fact_base["srk_tmp"].isna().sum())
missing_cia = int(df_fact_base["srk_cia"].isna().sum())
missing_apt = int(df_fact_base["srk_apt"].isna().sum())

if missing_tmp or missing_cia or missing_apt:
    raise ValueError(
        "Falha ao resolver SRKs:\n"
        f"- srk_tmp faltando: {missing_tmp}\n"
        f"- srk_cia faltando: {missing_cia}\n"
        f"- srk_apt faltando: {missing_apt}\n"
        "Isso acontece quando alguma chave natural da Silver não existe na DIM correspondente."
    )

# Montar a FATO (colunas do seu DDL)
df_fat_atr = pd.DataFrame({
    "srk_tmp": df_fact_base["srk_tmp"].astype("int64"),
    "srk_cia": df_fact_base["srk_cia"].astype("int64"),
    "srk_apt": df_fact_base["srk_apt"].astype("int64"),

    # gerais
    "qtd_voo_cgd": pd.to_numeric(df_fact_base["arr_flights"], errors="coerce").fillna(0).astype("int64"),
    "qtd_atr_ats": pd.to_numeric(df_fact_base["arr_del15"], errors="coerce").fillna(0).astype("int64"),
    "qtd_voo_can": pd.to_numeric(df_fact_base["arr_cancelled"], errors="coerce").fillna(0).astype("int64"),
    "qtd_voo_div": pd.to_numeric(df_fact_base["arr_diverted"], errors="coerce").fillna(0).astype("int64"),
    "val_atr_mnt": pd.to_numeric(df_fact_base["arr_delay"], errors="coerce").fillna(0).round(2),

    # contagens por causa
    "qtd_atr_cia":     pd.to_numeric(df_fact_base["carrier_ct"], errors="coerce").fillna(0).astype("int64"),
    "qtd_atr_cli":     pd.to_numeric(df_fact_base["weather_ct"], errors="coerce").fillna(0).astype("int64"),
    "qtd_atr_nas":     pd.to_numeric(df_fact_base["nas_ct"], errors="coerce").fillna(0).astype("int64"),
    "qtd_atr_seg":     pd.to_numeric(df_fact_base["security_ct"], errors="coerce").fillna(0).astype("int64"),
    "qtd_atr_aer_tar": pd.to_numeric(df_fact_base["late_aircraft_ct"], errors="coerce").fillna(0).astype("int64"),

    # minutos por causa
    "val_atr_cia_mnt":     pd.to_numeric(df_fact_base["carrier_delay"], errors="coerce").fillna(0).round(2),
    "val_atr_cli_mnt":     pd.to_numeric(df_fact_base["weather_delay"], errors="coerce").fillna(0).round(2),
    "val_atr_nas_mnt":     pd.to_numeric(df_fact_base["nas_delay"], errors="coerce").fillna(0).round(2),
    "val_atr_seg_mnt":     pd.to_numeric(df_fact_base["security_delay"], errors="coerce").fillna(0).round(2),
    "val_atr_aer_tar_mnt": pd.to_numeric(df_fact_base["late_aircraft_delay"], errors="coerce").fillna(0).round(2),

    # qualidade (na Silver continua sendo is_outlier_arr_delay)
    "ind_out_atr": df_fact_base["is_outlier_arr_delay"].fillna(False).astype(bool),
})

# Garantir não-negatividade para passar nos CHECKs do DDL (segurança)
num_cols_int = ["qtd_voo_cgd","qtd_atr_ats","qtd_voo_can","qtd_voo_div","qtd_atr_cia","qtd_atr_cli","qtd_atr_nas","qtd_atr_seg","qtd_atr_aer_tar"]
for c in num_cols_int:
    df_fat_atr[c] = df_fat_atr[c].clip(lower=0)

num_cols_num = ["val_atr_mnt","val_atr_cia_mnt","val_atr_cli_mnt","val_atr_nas_mnt","val_atr_seg_mnt","val_atr_aer_tar_mnt"]
for c in num_cols_num:
    df_fat_atr[c] = df_fat_atr[c].clip(lower=0).round(2)

# Deduplicação no grão da FATO (srk_tmp,srk_cia,srk_apt)
before = len(df_fat_atr)
df_fat_atr = df_fat_atr.drop_duplicates(subset=["srk_tmp", "srk_cia", "srk_apt"]).reset_index(drop=True)
after = len(df_fat_atr)

print(f"FATO: linhas antes={before}, depois dedup={after}")
display(df_fat_atr.head(5))


ETAPA 5: TRANSFORM - Resolver SRKs e montar FATO (DataFrame)
FATO: linhas antes=279182, depois dedup=279182


,srk_tmp,srk_cia,srk_apt,qtd_voo_cgd,qtd_atr_ats,qtd_voo_can,qtd_voo_div,val_atr_mnt,qtd_atr_cia,qtd_atr_cli,qtd_atr_nas,qtd_atr_seg,qtd_atr_aer_tar,val_atr_cia_mnt,val_atr_cli_mnt,val_atr_nas_mnt,val_atr_seg_mnt,val_atr_aer_tar_mnt,ind_out_atr
0,8,8,289,650,126,4,0,5425.0,21,6,52,1,46,881.0,397.0,2016.0,15.0,2116.0,False
1,8,8,290,314,61,30,3,2801.0,14,3,34,0,10,478.0,239.0,1365.0,0.0,719.0,False
2,8,8,295,513,97,15,0,4261.0,28,0,52,0,17,1150.0,16.0,2286.0,0.0,809.0,False
3,8,8,296,334,78,3,1,3400.0,20,2,39,0,16,1159.0,166.0,1295.0,0.0,780.0,False
4,8,8,302,217,47,4,1,1737.0,8,0,22,0,17,350.0,28.0,522.0,0.0,837.0,False


## Célula 7 — LOAD da tabela fato
Carrega o DataFrame final da fato (df_fat_atr) na tabela dw.fat_atr, inserindo apenas as colunas previstas no DDL e deixando o identificador serial (srk_fat) ser gerado pelo próprio banco. A inserção é feita em lote via to_sql para reduzir overhead, e ao final é realizada uma conferência objetiva comparando a contagem de registros persistidos na DW com o volume do DataFrame, garantindo aderência ao grão imposto pela UNIQUE(srk_tmp, srk_cia, srk_apt).

In [7]:
print("=" * 80)
print("ETAPA 6: LOAD - Inserir FATO na DW")
print("=" * 80)

fat_cols = [
    "srk_tmp", "srk_cia", "srk_apt",
    "qtd_voo_cgd", "qtd_atr_ats", "qtd_voo_can", "qtd_voo_div", "val_atr_mnt",
    "qtd_atr_cia", "qtd_atr_cli", "qtd_atr_nas", "qtd_atr_seg", "qtd_atr_aer_tar",
    "val_atr_cia_mnt", "val_atr_cli_mnt", "val_atr_nas_mnt", "val_atr_seg_mnt", "val_atr_aer_tar_mnt",
    "ind_out_atr",
]

df_fat_load = df_fat_atr[fat_cols].copy()

df_fat_load.to_sql(
    "fat_atr",
    con=engine,
    schema=DW_SCHEMA,
    if_exists="append",
    index=False,
    method="multi",
    chunksize=5000
)

with engine.connect() as conn:
    qtd_fat = conn.execute(text(f"SELECT COUNT(*) FROM {DW_SCHEMA}.fat_atr;")).scalar()

print(f"Linhas inseridas na FATO ({DW_SCHEMA}.fat_atr): {qtd_fat}")
print("Linhas no DataFrame da FATO:", len(df_fat_load))


ETAPA 6: LOAD - Inserir FATO na DW
Linhas inseridas na FATO (dw.fat_atr): 279182
Linhas no DataFrame da FATO: 279182
